# generate stats for visium bins 

# Load Modules 

In [5]:
from pathlib import Path
import sys
import importlib
import os
import anndata as an
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import squidpy as sq
import scanpy as sc
from matplotlib.patches import Patch

plt.rcdefaults()

# Load Finalized TMA_obj 

In [4]:
os.chdir("/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/data/Final_Objs")
TMA_obj_final = an.read_h5ad("final_TMA_obj_V3.h5ad")

# Load raw h5ad 

In [28]:
h5ad_dir = Path("/home/plopez/orcd/pool/Projects/Preg_placenta_VISIUM/data/raw_h5ad_objs")

h5ad_files = [
    "TMA2140_obj_b2c_V1.h5ad",
    "TMA2145_obj_b2c_V1.h5ad",
    "TMA4_obj_b2c_V1.h5ad",
    "TMA9_obj_b2c_V1.h5ad",
]

renaming_dict = {
    "TMA2140":"tma2_140",
    "TMA2145":"tma2_145",
    "TMA4":"tma4",
    "TMA9":"tma9",
}

adatas = {}

for f in h5ad_files:
    sample_id = f.replace("_obj_b2c_V1.h5ad", "")
    
    a = sc.read_h5ad(h5ad_dir / f)
    a.obs_names = a.obs_names.astype(str)+"-"+renaming_dict[sample_id]
    
    adatas[sample_id] = a

TMA_obj_raw = an.concat(
    adatas,
    join="inner",
    label="sample_id_concat",
    index_unique=None,
    merge="same"
)


In [19]:
TMA_obj_final

AnnData object with n_obs × n_vars = 317507 × 10964
    obs: 'in_tissue', 'array_row', 'array_col', 'Row_num', 'Tissue_core', 'Patient', 'Condition', 'TMA_num', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_counts', 'library_id', '_indices', '_scvi_batch', '_scvi_labels', 'CD14_monocyte', 'CD16_monocyte', 'EVT1', 'EVT2', 'EVT3', 'EVT4', 'EVT5', 'EVT6', 'Endo_f1', 'Endo_f2', 'Endo_f3', 'Endo_f4', 'Endo_fProlif', 'Endo_m1', 'Endo_m2', 'Endo_m3', 'Endo_mL', 'HBC_1', 'HBC_10_cycling', 'HBC_11', 'HBC_2_CXCL2_CXCL3', 'HBC_3', 'HBC_4_IL1B_MMP19', 'HBC_5', 'HBC_6', 'HBC_7_lnRNA', 'HBC_8_neuro', 'HBC_9_CXCL10_IFIT44', 'HSC', 'HSC_cycling', 'MAST', 'Nuetrophils', 'Platelets', 'SCT_1', 'VCT_1', 'VCT_2', 'VCT_3', 'VCT_4', 'VCTccc_1', 'VCTfusion_1', 'VCTfusion_2', 'VCTp_1', '

# Calculate metric 

In [27]:
TMA_obj_raw.var

,gene_ids,feature_types,genome,mt,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts
SAMD11,ENSG00000187634,Gene Expression,GRCh38,False,97,0.000315,0.000315,99.969463,100.0,4.615120
NOC2L,ENSG00000188976,Gene Expression,GRCh38,False,5898,0.019417,0.019231,98.143251,6168.0,8.727292
KLHL17,ENSG00000187961,Gene Expression,GRCh38,False,1379,0.004508,0.004498,99.565877,1432.0,7.267526
PLEKHN1,ENSG00000187583,Gene Expression,GRCh38,False,302,0.000966,0.000966,99.904927,307.0,5.730100
PERM1,ENSG00000187642,Gene Expression,GRCh38,False,110,0.000359,0.000359,99.965371,114.0,4.744932
...,...,...,...,...,...,...,...,...,...,...
MT-ND4L,ENSG00000212907,Gene Expression,GRCh38,True,208562,1.714175,0.998488,34.342614,544511.0,13.207645
MT-ND4,ENSG00000198886,Gene Expression,GRCh38,True,231057,2.120245,1.137911,27.260965,673500.0,13.420245
MT-ND5,ENSG00000198786,Gene Expression,GRCh38,True,59846,0.234184,0.210410,81.159886,74389.0,11.217077
MT-ND6,ENSG00000198695,Gene Expression,GRCh38,True,45005,0.164010,0.151871,85.831980,52098.0,10.860901


In [ ]:
# ----------------------------
# compute QC metrics on the truly raw object
# ----------------------------
TMA_obj_raw_all.var["mt"] = TMA_obj_raw_all.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(TMA_obj_raw_all, qc_vars=["mt"], inplace=True)

# stage 2: min_counts >= 200
TMA_obj_min200 = TMA_obj_raw_all[TMA_obj_raw_all.obs["total_counts"] >= 200, :].copy()

# stage 3: mt < 20
TMA_obj_final_from_raw = TMA_obj_min200[TMA_obj_min200.obs["pct_counts_mt"] < 20, :].copy()

# optional check against your finalized object
TMA_obj_final = an.read_h5ad(final_obj_path)
print("final_from_raw:", TMA_obj_final_from_raw.n_obs, TMA_obj_final_from_raw.n_vars)
print("loaded_final:  ", TMA_obj_final.n_obs, TMA_obj_final.n_vars)

# ----------------------------
# helper to summarize one object per TMA core
# ----------------------------
group_cols = ["Row_num", "Tissue_core", "Patient", "Condition"]

def summarize_qc(adata, stage_name):
    df = adata.obs.copy()

    # make sure all expected QC columns exist
    needed = [
        "n_genes_by_counts",
        "total_counts",
        "total_counts_mt",
        "pct_counts_mt",
    ]
    for col in needed:
        if col not in df.columns:
            raise ValueError(f"Missing expected QC column: {col}")

    out = (
        df.groupby(group_cols, observed=True)
        .agg(
            n_bins=("Row_num", "size"),
            median_genes_per_bin=("n_genes_by_counts", "median"),
            mean_genes_per_bin=("n_genes_by_counts", "mean"),
            median_UMIs_per_bin=("total_counts", "median"),
            mean_UMIs_per_bin=("total_counts", "mean"),
            total_UMIs=("total_counts", "sum"),
            median_mt_pct=("pct_counts_mt", "median"),
            mean_mt_pct=("pct_counts_mt", "mean"),
            total_mt_UMIs=("total_counts_mt", "sum"),
        )
        .reset_index()
    )

    out = out.rename(
        columns={
            "n_bins": f"n_bins__{stage_name}",
            "median_genes_per_bin": f"median_genes_per_bin__{stage_name}",
            "mean_genes_per_bin": f"mean_genes_per_bin__{stage_name}",
            "median_UMIs_per_bin": f"median_UMIs_per_bin__{stage_name}",
            "mean_UMIs_per_bin": f"mean_UMIs_per_bin__{stage_name}",
            "total_UMIs": f"total_UMIs__{stage_name}",
            "median_mt_pct": f"median_mt_pct__{stage_name}",
            "mean_mt_pct": f"mean_mt_pct__{stage_name}",
            "total_mt_UMIs": f"total_mt_UMIs__{stage_name}",
        }
    )

    return out

# ----------------------------
# summarize all stages
# ----------------------------
qc_raw = summarize_qc(TMA_obj_raw_all, "raw_all_bins")
qc_min200 = summarize_qc(TMA_obj_min200, "after_min_counts200")
qc_mt20 = summarize_qc(TMA_obj_final_from_raw, "after_mt20")

# ----------------------------
# merge into one supplementary-style table
# ----------------------------
qc = meta.merge(qc_raw, on=group_cols, how="left")
qc = qc.merge(qc_min200, on=group_cols, how="left")
qc = qc.merge(qc_mt20, on=group_cols, how="left")

# ----------------------------
# add retention / filtering summary columns
# ----------------------------
qc["bins_removed_by_min_counts"] = (
    qc["n_bins__raw_all_bins"] - qc["n_bins__after_min_counts200"]
)

qc["bins_removed_by_mt20"] = (
    qc["n_bins__after_min_counts200"] - qc["n_bins__after_mt20"]
)

qc["pct_bins_retained_after_min_counts"] = (
    100 * qc["n_bins__after_min_counts200"] / qc["n_bins__raw_all_bins"]
)

qc["pct_bins_retained_after_mt20"] = (
    100 * qc["n_bins__after_mt20"] / qc["n_bins__after_min_counts200"]
)

qc["pct_bins_retained_final_vs_raw"] = (
    100 * qc["n_bins__after_mt20"] / qc["n_bins__raw_all_bins"]
)

qc["mt_filter_rule"] = "pct_counts_mt < 20"
qc["count_filter_rule"] = "total_counts >= 200"

# ----------------------------
# optional nicer column ordering
# ----------------------------
front_cols = [
    "Block_num",
    "Name_on_visium",
    "TMA_num",
    "Row",
    "Row_num",
    "Tissue_core",
    "Patient",
    "Condition",
    "count_filter_rule",
    "mt_filter_rule",
    "n_bins__raw_all_bins",
    "n_bins__after_min_counts200",
    "n_bins__after_mt20",
    "bins_removed_by_min_counts",
    "bins_removed_by_mt20",
    "pct_bins_retained_after_min_counts",
    "pct_bins_retained_after_mt20",
    "pct_bins_retained_final_vs_raw",
]

other_cols = [c for c in qc.columns if c not in front_cols]
qc = qc[front_cols + other_cols]

# ----------------------------
# round for presentation
# ----------------------------
round_cols = [c for c in qc.columns if any(x in c for x in ["median", "mean", "pct_"])]
qc[round_cols] = qc[round_cols].round(2)

# ----------------------------
# export
# ----------------------------
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    qc.to_excel(writer, sheet_name="VisiumHD_QC_by_core", index=False)
    meta.to_excel(writer, sheet_name="metadata_source", index=False)

print(f"Saved: {out_path}")
qc.head()